# Ensemble Analysis — Technique 1 (Pooled-Block)

This notebook introduces **Technique 1** for ensemble UQ analysis in `quends`. Technique 1 is the **pooled-block** method: it computes short-time (block) averages from each ensemble member and then pools those block averages together to produce a combined estimate.

**When to use Technique 1:**
- Ensemble members have different run lengths.
- You want to weight each member equally regardless of length.
- You need a robust estimate that is less sensitive to a single long run dominating the result.

**Technique comparison overview:**

| Technique | Name | Description |
|-----------|------|-------------|
| 0 | Average-then-analyze | Pool all member data into one stream, analyze combined |
| **1** | **Pooled-block** | **Block-average each member, pool the blocks** |
| 2 | Member-wise IVW | Analyze each member independently, combine with inverse-variance weighting |

## 1. Load Ensemble Members

We load three simulation runs (members) of the same physical scenario from CSV files. In this example the files correspond to GX gyrokinetic turbulence simulations with `tprim = 2.5`, run with different random seeds to capture statistical variability.

In [ ]:
import quends as qnds

files = [
    "gx/ensemble/tprim_2_5_a.out.csv",
    "gx/ensemble/tprim_2_5_b.out.csv",
    "gx/ensemble/tprim_2_5_c.out.csv",
]

members = [qnds.from_csv(f) for f in files]
ens = qnds.Ensemble(members)

print(f"Ensemble loaded with {len(ens.data_streams)} members.")

## 2. Inspect Member Data

Before trimming, it is good practice to inspect the first few rows of each member to verify the data loaded correctly and to understand the column names available.

In [ ]:
for i, ds in enumerate(ens.data_streams):
    print(f"\n--- Member {i} ---")
    print(f"  Variables : {ds.variables()}")
    print(f"  Time steps: {len(ds.data)}")
    print(ds.head())

## 3. Trim the Ensemble

Plasma turbulence simulations have a transient startup phase that must be removed before statistical analysis. We use `Ensemble.trim()` to apply the same trimming strategy to every member automatically.

Here we use `method="std"` (the `QuantileTrimStrategy`) with a `window_size=20`. This strategy uses a rolling-window standard deviation to detect where the signal has settled into a stationary state.

In [ ]:
trimmed_ens = ens.trim("HeatFlux_st", method="std", window_size=20)

for i, ds in enumerate(trimmed_ens.data_streams):
    print(f"Member {i}: {len(ds.data)} time steps after trimming")

## 4. Compute Statistics — Technique 1 (Pooled-Block)

Technique 1 works as follows:

1. Each member's stationary region is divided into non-overlapping blocks.
2. A mean is computed for each block (short-time averages).
3. All block means across all members are pooled together.
4. A grand mean and confidence interval are computed from this pool.

This approach naturally handles members with **different lengths** because the block structure normalizes member contributions.

In [ ]:
results = trimmed_ens.compute_statistics("HeatFlux_st", technique=1)
print(results)

## 5. Compare Technique 0 vs Technique 1

To appreciate what Technique 1 adds, we compare it directly against Technique 0 (average-then-analyze), which simply concatenates all member data into a single stream.

**Key differences:**

- **Technique 0** is the simplest approach. If one member is much longer than the others, it will disproportionately influence the result.
- **Technique 1** mitigates length bias by block-averaging, so each member contributes more equally.

In many gyrokinetic turbulence workflows the two techniques give similar means but can differ in confidence interval width.

In [ ]:
for technique in [0, 1]:
    r = trimmed_ens.compute_statistics("HeatFlux_st", technique=technique)
    label = "Average-then-analyze" if technique == 0 else "Pooled-block       "
    print(f"Technique {technique} ({label}): mean={r['mean']:.4f}, CI={r['confidence_interval']}")

## Summary

- **Technique 1 (Pooled-Block)** is well-suited for ensembles whose members have different run lengths.
- It creates block averages within each member before pooling, which reduces the influence of longer runs.
- Use `Ensemble.trim()` with a `method` and `window_size` to prepare the ensemble before calling `compute_statistics()`.
- The `technique=1` argument selects the pooled-block algorithm.

**Next:** See `05_technique2_ensemble.ipynb` for the most rigorous approach — member-wise inverse-variance weighting (Technique 2).